# Dataset v6 — COVID-19 UCI: Mortalidad a 30 días

**Cambios respecto a v5:**
- Extracción de nuevas variables de laboratorio: pH arterial (100339), Na, K, Hematocrito, WBC, PEEP, Volumen Tidal
- Cálculo de **SOFA parcial** (5/6 componentes — sin GCS por no disponibilidad en datos estructurados)
- Cálculo de **APACHE II parcial** (11/12 componentes — sin GCS ni puntos por enfermedad crónica)
- Actualización del severity_score incluyendo las nuevas variables

**Arquitectura del notebook:**
1. [Spark] Extracción y joins sobre datos raw
2. [Spark] Generación del parquet all_stays con todas las features
3. [Pandas] Cálculo de SOFA parcial y APACHE II parcial
4. [Pandas] Indicadoras de missingness, severity_score e imputación
5. [Pandas] Compactación OptionB / OptionC y guardado

## 1. Setup

In [5]:
import os
import sys
os.environ['_JAVA_OPTIONS'] = '-Djava.security.manager=allow -Duser.name=julianromero'
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from pyspark.sql import SparkSession
import pyspark.pandas as ps

spark = SparkSession.builder.appName('SparkSession').getOrCreate()

from utils.sql_manager import SQLManager
sql_manager = SQLManager(queries_dir='../queries')

## 2. Carga de archivos raw

In [6]:
hosp_adm        = spark.read.csv('../data/nw_hosp/admissions.csv',     header=True, inferSchema=True)
icu_stays       = spark.read.csv('../data/nw_icu/icustays.csv',        header=True, inferSchema=True)
patients        = spark.read.csv('../data/nw_hosp/patients.csv',       header=True, inferSchema=True)
chart_events    = spark.read.csv('../data/nw_icu/chartevents.csv',     header=True, inferSchema=True)
lab_events      = spark.read.csv('../data/nw_hosp/labevents.csv',      header=True, inferSchema=True)
procedure_events= spark.read.csv('../data/nw_icu/procedureevents.csv', header=True, inferSchema=True)
diagnoses_icd   = spark.read.csv('../data/nw_hosp/diagnoses_icd.csv',  header=True, inferSchema=True)

## 3. Vistas temporales

In [7]:
hosp_adm.createOrReplaceTempView('admissions')
icu_stays.createOrReplaceTempView('icu_stays')
patients.createOrReplaceTempView('patients')
chart_events.createOrReplaceTempView('chart_events')
lab_events.createOrReplaceTempView('lab_events')
procedure_events.createOrReplaceTempView('procedure_events')
diagnoses_icd.createOrReplaceTempView('diagnoses_icd')

## 4. Datos iniciales (queries SQL)

In [8]:
hosp_adm_data = sql_manager.execute(spark, 'initial_data/admissions.sql')
hosp_adm_data.createOrReplaceTempView('hosp_adm_data')

icu_stays_data = sql_manager.execute(spark, 'initial_data/icu_stays.sql')
icu_stays_data.createOrReplaceTempView('icu_stays_data')

chart_events_data = sql_manager.execute(spark, 'initial_data/chart_events.sql')
chart_events_data.createOrReplaceTempView('chart_events_data')

patients_data = sql_manager.execute(spark, 'initial_data/patients_filtered.sql')
patients_data.createOrReplaceTempView('patients_data')

lab_events_data = sql_manager.execute(spark, 'initial_data/lab_events.sql')
lab_events_data.createOrReplaceTempView('lab_events_data')

procedure_events_data = sql_manager.execute(spark, 'initial_data/procedure_events.sql')
procedure_events_data.createOrReplaceTempView('procedure_events_data')

## 5. Joins ICU stays → eventos (ventana 24h desde admisión)

In [9]:
# JOIN: ICU stays x chart_events (primeras 24h)
chart_events_icu = sql_manager.execute(spark, 'joins/icu_stays_chartevents.sql')
chart_events_icu.createOrReplaceTempView('chart_events_icu')

# JOIN: ICU stays x lab_events (primeras 24h)
lab_events_icu = sql_manager.execute(spark, 'joins/icu_stays_labevents.sql')
lab_events_icu.createOrReplaceTempView('lab_events_icu')

# JOIN: ICU stays x procedure_events (primeras 24h)
procedure_events_icu = sql_manager.execute(spark, 'joins/icu_stays_procedure_events.sql')
procedure_events_icu.createOrReplaceTempView('procedure_events_icu')

print(f'chart_events_icu: {chart_events_icu.count()} registros')
print(f'lab_events_icu:   {lab_events_icu.count()} registros')
print(f'procedure_events_icu: {procedure_events_icu.count()} registros')

chart_events_icu: 1676730 registros


lab_events_icu:   573331 registros
procedure_events_icu: 275181 registros


## 6. Extracción de features de chart_events

Signos vitales: temperatura, PA (invasiva y no invasiva), SpO₂, pulso.

In [10]:
# Temperatura: max en ventana (itemid 323761, unidad °F)
# Rango clínico esperado: 86–113 °F (30–45 °C)
temperature = spark.sql("""
    SELECT subject_id, admission_id, stay_id,
           MAX(try_cast(value AS DOUBLE)) AS max_temp_f
    FROM chart_events_icu
    WHERE itemid = 323761
      AND value IS NOT NULL
      AND try_cast(value AS DOUBLE) BETWEEN 86 AND 113
    GROUP BY subject_id, admission_id, stay_id
""")
temperature.createOrReplaceTempView('temperature')
print(f'temperature: {temperature.count()} stays')

temperature: 19525 stays


In [11]:
# PA invasiva (línea arterial): min sistólica (320050) y diastólica (320051)
sbp_line = spark.sql("""
    SELECT subject_id, admission_id, stay_id,
           MIN(try_cast(value AS DOUBLE)) AS min_bp_systolic_line
    FROM chart_events_icu
    WHERE itemid = 320050
      AND value IS NOT NULL
      AND try_cast(value AS DOUBLE) > 0
    GROUP BY subject_id, admission_id, stay_id
""")
dbp_line = spark.sql("""
    SELECT subject_id, admission_id, stay_id,
           MIN(try_cast(value AS DOUBLE)) AS min_bp_diastolic_line
    FROM chart_events_icu
    WHERE itemid = 320051
      AND value IS NOT NULL
      AND try_cast(value AS DOUBLE) > 0
    GROUP BY subject_id, admission_id, stay_id
""")
sbp_line.createOrReplaceTempView('sbp_line')
dbp_line.createOrReplaceTempView('dbp_line')

bp_line = sbp_line.join(dbp_line, on=['subject_id', 'admission_id', 'stay_id'], how='full')
bp_line.createOrReplaceTempView('bp_line')

# PA no invasiva: min sistólica (320179) y diastólica (320180)
sbp_noninv = spark.sql("""
    SELECT subject_id, admission_id, stay_id,
           MIN(try_cast(value AS DOUBLE)) AS min_bp_systolic
    FROM chart_events_icu
    WHERE itemid = 320179
      AND value IS NOT NULL
      AND try_cast(value AS DOUBLE) > 0
    GROUP BY subject_id, admission_id, stay_id
""")
dbp_noninv = spark.sql("""
    SELECT subject_id, admission_id, stay_id,
           MIN(try_cast(value AS DOUBLE)) AS min_bp_diastolic
    FROM chart_events_icu
    WHERE itemid = 320180
      AND value IS NOT NULL
      AND try_cast(value AS DOUBLE) > 0
    GROUP BY subject_id, admission_id, stay_id
""")
sbp_noninv.createOrReplaceTempView('sbp_noninv')
dbp_noninv.createOrReplaceTempView('dbp_noninv')

bp_noninv = sbp_noninv.join(dbp_noninv, on=['subject_id', 'admission_id', 'stay_id'], how='full')
bp_noninv.createOrReplaceTempView('bp_noninv')

all_bp = bp_line.join(bp_noninv, on=['subject_id', 'admission_id', 'stay_id'], how='full')
all_bp.createOrReplaceTempView('all_bp')
print(f'all_bp: {all_bp.count()} stays')

all_bp: 20011 stays


In [12]:
# SpO2: mín (itemid 320277)
sp_o2 = spark.sql("""
    SELECT subject_id, admission_id, stay_id,
           MIN(try_cast(value AS DOUBLE)) AS min_oxygen_saturation
    FROM chart_events_icu
    WHERE itemid = 320277
      AND value IS NOT NULL
      AND try_cast(value AS DOUBLE) BETWEEN 50 AND 100
    GROUP BY subject_id, admission_id, stay_id
""")
sp_o2.createOrReplaceTempView('sp_o2')

# FC: max (itemid 320045)
pulse = spark.sql("""
    SELECT subject_id, admission_id, stay_id,
           MAX(try_cast(value AS DOUBLE)) AS max_pulse
    FROM chart_events_icu
    WHERE itemid = 320045
      AND value IS NOT NULL
      AND try_cast(value AS DOUBLE) BETWEEN 20 AND 300
    GROUP BY subject_id, admission_id, stay_id
""")
pulse.createOrReplaceTempView('pulse')

# FR: max (itemid 320210)
resp_rate = spark.sql("""
    SELECT subject_id, admission_id, stay_id,
           MAX(try_cast(value AS DOUBLE)) AS max_resp_rate
    FROM chart_events_icu
    WHERE itemid = 320210
      AND value IS NOT NULL
      AND try_cast(value AS DOUBLE) BETWEEN 4 AND 70
    GROUP BY subject_id, admission_id, stay_id
""")
resp_rate.createOrReplaceTempView('resp_rate')

# BMI (itemid 300001)
bmi = spark.sql("""
    SELECT subject_id, admission_id, stay_id,
           MAX(try_cast(value AS DOUBLE)) AS bmi
    FROM chart_events_icu
    WHERE itemid = 300001
      AND value IS NOT NULL
      AND try_cast(value AS DOUBLE) BETWEEN 10 AND 80
    GROUP BY subject_id, admission_id, stay_id
""")
bmi.createOrReplaceTempView('bmi')

print('Vitales extraídos: SpO2, FC, FR, BMI')

Vitales extraídos: SpO2, FC, FR, BMI


## 7. Extracción de features de lab_events (v6)

**Nuevos itemids respecto a v5:**
| Variable | itemid | Uso |
|---|---|---|
| pH arterial | 100339 | APACHE II componente 6 |
| Sodio | 100010 | APACHE II componente 7 |
| Potasio | 100011 | APACHE II componente 8 |
| Hematocrito | 100006 | APACHE II componente 10 |
| WBC | 100016 | APACHE II componente 11 |
| PEEP | 100060 | Estimación FiO₂ para ratio PF |
| Volumen tidal | 100077 | Contexto ventilatorio |

In [13]:
lab_features_v6 = spark.sql("""
    SELECT subject_id, admission_id, stay_id,

        -- ── SOFA: componentes disponibles ──────────────────────────────────
        -- Respiratorio
        MIN(CASE WHEN itemid = 100029 THEN value_num END) AS min_pao2,
        -- Coagulación
        MIN(CASE WHEN itemid = 100014 THEN value_num END) AS min_platelets,
        -- Hepático
        MAX(CASE WHEN itemid = 100020 THEN value_num END) AS max_bilirubin,
        -- Renal
        MAX(CASE WHEN itemid = 100002 THEN value_num END) AS max_creatinine,
        -- (Cardiovascular: MAP desde chartevents; Neurológico: GCS no disponible)

        -- ── APACHE II: nuevas variables (v6) ───────────────────────────────
        MIN(CASE WHEN itemid = 100339 THEN value_num END) AS min_ph,
        MAX(CASE WHEN itemid = 100339 THEN value_num END) AS max_ph,
        MIN(CASE WHEN itemid = 100010 THEN value_num END) AS min_sodium,
        MAX(CASE WHEN itemid = 100010 THEN value_num END) AS max_sodium,
        MIN(CASE WHEN itemid = 100011 THEN value_num END) AS min_potassium,
        MAX(CASE WHEN itemid = 100011 THEN value_num END) AS max_potassium,
        MIN(CASE WHEN itemid = 100006 THEN value_num END) AS min_hematocrit,
        MAX(CASE WHEN itemid = 100006 THEN value_num END) AS max_hematocrit,
        MIN(CASE WHEN itemid = 100016 THEN value_num END) AS min_wbc,
        MAX(CASE WHEN itemid = 100016 THEN value_num END) AS max_wbc,

        -- ── Parámetros ventilatorios (para estimar FiO₂) ───────────────────
        MAX(CASE WHEN itemid = 100060 THEN value_num END) AS max_peep,
        MAX(CASE WHEN itemid = 100077 THEN value_num END) AS max_tidal_volume,

        -- ── Otros labs clínicamente relevantes ────────────────────────────
        MAX(CASE WHEN itemid = 100031 THEN value_num END) AS max_lactate,
        MAX(CASE WHEN itemid = 100004 THEN value_num END) AS max_bun,
        MAX(CASE WHEN itemid = 100001 THEN value_num END) AS max_glucose,
        MAX(CASE WHEN itemid = 100034 THEN value_num END) AS max_pt,
        MAX(CASE WHEN itemid = 100053 THEN value_num END) AS max_crp,
        MAX(CASE WHEN itemid = 100052 THEN value_num END) AS max_ferritin,
        MAX(CASE WHEN itemid = 100075 THEN value_num END) AS max_dimer,
        MIN(CASE WHEN itemid = 100037 THEN value_num END) AS min_lymphocytes,
        MAX(CASE WHEN itemid = 100022 THEN value_num END) AS max_neutrophils,
        MAX(CASE WHEN itemid = 100059 THEN value_num END) AS max_troponin

    FROM lab_events_icu
    GROUP BY subject_id, admission_id, stay_id
""")

lab_features_v6.createOrReplaceTempView('lab_features_v6')
print(f'lab_features_v6: {lab_features_v6.count()} stays | {len(lab_features_v6.columns)} columnas')
lab_features_v6.show(3)

26/04/03 13:21:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


lab_features_v6: 21253 stays | 29 columnas


+----------+------------+--------+--------+-------------+-------------+--------------+------+------+----------+----------+-------------+-------------+--------------+--------------+-------+-------+--------+----------------+-----------+-------+-----------+------+-------+------------+---------+---------------+---------------+------------+
|subject_id|admission_id| stay_id|min_pao2|min_platelets|max_bilirubin|max_creatinine|min_ph|max_ph|min_sodium|max_sodium|min_potassium|max_potassium|min_hematocrit|max_hematocrit|min_wbc|max_wbc|max_peep|max_tidal_volume|max_lactate|max_bun|max_glucose|max_pt|max_crp|max_ferritin|max_dimer|min_lymphocytes|max_neutrophils|max_troponin|
+----------+------------+--------+--------+-------------+-------------+--------------+------+------+----------+----------+-------------+-------------+--------------+--------------+-------+-------+--------+----------------+-----------+-------+-----------+------+-------+------------+---------+---------------+---------------+

### 7.1 Cobertura de nuevas variables (verificación)

In [14]:
from pyspark.sql import functions as F

new_cols = ['min_ph', 'min_sodium', 'min_potassium', 'min_hematocrit',
            'min_wbc', 'max_peep', 'max_tidal_volume']

total = lab_features_v6.count()
print('Cobertura de variables nuevas (% no-nulos):')
for col in new_cols:
    non_null = lab_features_v6.filter(F.col(col).isNotNull()).count()
    print(f'  {col:<25}: {non_null/total:.1%} ({non_null}/{total})')

Cobertura de variables nuevas (% no-nulos):


  min_ph                   : 0.0% (0/21253)


  min_sodium               : 64.1% (13614/21253)


  min_potassium            : 64.1% (13626/21253)


  min_hematocrit           : 53.2% (11307/21253)


  min_wbc                  : 52.3% (11124/21253)


  max_peep                 : 9.5% (2025/21253)


  max_tidal_volume         : 6.3% (1332/21253)


## 8. Flags de procedimientos

In [15]:
procedure_flags = sql_manager.execute(spark, 'joins/procedure_flags.sql')
procedure_flags.createOrReplaceTempView('procedure_flags')
print(f'procedure_flags: {procedure_flags.count()} stays')
procedure_flags.show(3)

procedure_flags: 20567 stays


+----------+------------+--------+-------------------------+-------+-----------------+------------+-----------------+--------------+------+----------------+----------------+-------+-------+
|subject_id|admission_id| stay_id|is_mechanical_ventilation|is_crrt|has_arterial_line|is_intubated|is_prone_position|has_chest_tube|is_dnr|has_central_line|has_hemodialysis|has_niv|is_ecmo|
+----------+------------+--------+-------------------------+-------+-----------------+------------+-----------------+--------------+------+----------------+----------------+-------+-------+
|  30021684|    42899119|58225653|                        0|      0|                0|           0|                1|             0|     0|               0|               0|      0|      0|
|  30123896|    42219130|54862620|                        0|      0|                0|           0|                1|             0|     0|               0|               0|      0|      0|
|  30173978|    48100428|52762289|                

## 9. Construcción del dataset all_stays (Spark)

Criterio de inclusión: pacientes con diagnóstico COVID-19 (ICD-10: U071), todas las estancias.
La compactación a una estancia por paciente se realiza después en la fase pandas.

In [16]:
base_stays_v6 = spark.sql("""
    SELECT i.subject_id, i.admission_id, i.stay_id,
           i.length_of_stay, i.admittime
    FROM icu_stays_data i
    INNER JOIN diagnoses_icd d
        ON i.subject_id = d.subject_id
       AND i.admission_id = d.hadm_id
    WHERE d.icd_code = 'U071'
""")
base_stays_v6.createOrReplaceTempView('base_stays_v6')
print(f'Total stays COVID-19: {base_stays_v6.count()}')

Total stays COVID-19: 3908


In [17]:
PROCEDURE_COLS = [
    'is_mechanical_ventilation', 'is_crrt', 'has_arterial_line',
    'is_intubated', 'is_prone_position', 'has_chest_tube',
    'is_dnr', 'has_central_line', 'has_hemodialysis',
    'has_niv', 'is_ecmo'
]

dataset_v6_all = (
    base_stays_v6
    .join(all_bp,           on=['subject_id', 'admission_id', 'stay_id'], how='left')
    .join(sp_o2,            on=['subject_id', 'admission_id', 'stay_id'], how='left')
    .join(pulse,            on=['subject_id', 'admission_id', 'stay_id'], how='left')
    .join(resp_rate,        on=['subject_id', 'admission_id', 'stay_id'], how='left')
    .join(temperature,      on=['subject_id', 'admission_id', 'stay_id'], how='left')
    .join(bmi,              on=['subject_id', 'admission_id', 'stay_id'], how='left')
    .join(lab_features_v6,  on=['subject_id', 'admission_id', 'stay_id'], how='left')
    .join(procedure_flags,  on=['subject_id', 'admission_id', 'stay_id'], how='left')
    .join(
        patients_data.select('subject_id', 'admission_id', 'gender', 'anchor_age', 'dod_within_30_days'),
        on=['subject_id', 'admission_id'], how='left'
    )
    .join(
        hosp_adm_data.select('subject_id', 'admission_id', 'marital_status', 'race'),
        on=['subject_id', 'admission_id'], how='left'
    )
)

# Rellenar NaN en flags de procedimientos
for col_name in PROCEDURE_COLS:
    dataset_v6_all = dataset_v6_all.fillna({col_name: 0})

dataset_v6_all.createOrReplaceTempView('dataset_v6_all')
print(f'dataset_v6_all: {dataset_v6_all.count()} filas | {len(dataset_v6_all.columns)} columnas')

dataset_v6_all: 3908 filas | 56 columnas


In [18]:
# Guardar parquet all_stays como punto de checkpoint
import os
os.makedirs('../data/processed', exist_ok=True)

dataset_v6_all.write.mode('overwrite').parquet('../data/processed/covid_icu_dataset_v6_all_stays.parquet')
print('Guardado: covid_icu_dataset_v6_all_stays.parquet')

Guardado: covid_icu_dataset_v6_all_stays.parquet


---
## 10. Post-procesamiento en Pandas

A partir de aquí se usa pandas. El parquet generado en el paso anterior es el punto de entrada.

In [19]:
import pandas as pd
import numpy as np
import os

os.makedirs('datasets/v6', exist_ok=True)

print('Cargando parquet all_stays...')
df = pd.read_parquet('../data/processed/covid_icu_dataset_v6_all_stays.parquet')
print(f'Shape: {df.shape}')
print(f'Columnas: {list(df.columns)}')

Cargando parquet all_stays...
Shape: (3908, 56)
Columnas: ['subject_id', 'admission_id', 'stay_id', 'length_of_stay', 'admittime', 'min_bp_systolic_line', 'min_bp_diastolic_line', 'min_bp_systolic', 'min_bp_diastolic', 'min_oxygen_saturation', 'max_pulse', 'max_resp_rate', 'max_temp_f', 'bmi', 'min_pao2', 'min_platelets', 'max_bilirubin', 'max_creatinine', 'min_ph', 'max_ph', 'min_sodium', 'max_sodium', 'min_potassium', 'max_potassium', 'min_hematocrit', 'max_hematocrit', 'min_wbc', 'max_wbc', 'max_peep', 'max_tidal_volume', 'max_lactate', 'max_bun', 'max_glucose', 'max_pt', 'max_crp', 'max_ferritin', 'max_dimer', 'min_lymphocytes', 'max_neutrophils', 'max_troponin', 'is_mechanical_ventilation', 'is_crrt', 'has_arterial_line', 'is_intubated', 'is_prone_position', 'has_chest_tube', 'is_dnr', 'has_central_line', 'has_hemodialysis', 'has_niv', 'is_ecmo', 'gender', 'anchor_age', 'dod_within_30_days', 'marital_status', 'race']


## 11. Cálculo del SOFA parcial (5/6 componentes)

**Componentes disponibles:**
| Componente | Variable | Fuente |
|---|---|---|
| Respiratorio | PaO₂/FiO₂ o SpO₂/FiO₂ proxy | lab + chart |
| Coagulación | Plaquetas | lab |
| Hepático | Bilirrubina | lab |
| Cardiovascular | PAM (calculada) | chart |
| Neurológico | **GCS — NO DISPONIBLE** | — |
| Renal | Creatinina | lab |

> **Nota metodológica:** El componente cardiovascular del SOFA original requiere dosis de vasopresores (dopamina, noradrenalina). Sin esa información, se puntúa 0 si PAM ≥ 70 y 1 si PAM < 70 (nivel mínimo). Esta limitación se documenta en la sección de limitaciones del TFG.

In [20]:
def compute_map(row):
    """PAM = (PAS + 2*PAD) / 3. Prioriza medición no invasiva; fallback a línea arterial."""
    sbp = row.get('min_bp_systolic')
    dbp = row.get('min_bp_diastolic')
    if pd.notna(sbp) and pd.notna(dbp):
        return (sbp + 2 * dbp) / 3
    sbp_l = row.get('min_bp_systolic_line')
    dbp_l = row.get('min_bp_diastolic_line')
    if pd.notna(sbp_l) and pd.notna(dbp_l):
        return (sbp_l + 2 * dbp_l) / 3
    return np.nan


def compute_sofa_partial(row):
    """
    SOFA score parcial (sin componente neurológico por ausencia de GCS).
    Máximo teórico: 20 puntos (4 por componente × 5 componentes).
    
    Returns: (score, n_components_disponibles)
    """
    score = 0
    n = 0

    # ── 1. RESPIRATORIO ────────────────────────────────────────────────────
    # Ratio PaO₂/FiO₂. FiO₂ estimada: ventilado=0.5, espontáneo=0.21
    # Si no hay PaO₂, se usa proxy SF (SpO₂/FiO₂): SF<235 ≈ PF<200, SF<315 ≈ PF<300
    is_ventilated = row.get('is_mechanical_ventilation', 0) == 1
    fio2 = 0.50 if is_ventilated else 0.21
    pf = None

    if pd.notna(row.get('min_pao2')) and row['min_pao2'] > 0:
        pf = row['min_pao2'] / fio2
    elif pd.notna(row.get('min_oxygen_saturation')):
        # SF proxy (Rice et al., 2007): SF ratio como sustituto del PF
        pf = row['min_oxygen_saturation'] / fio2

    if pf is not None:
        n += 1
        if pf >= 400:   score += 0
        elif pf >= 300: score += 1
        elif pf >= 200: score += 2
        elif pf >= 100: score += 3
        else:           score += 4

    # ── 2. COAGULACIÓN (plaquetas ×10³/µL) ────────────────────────────────
    plt = row.get('min_platelets')
    if pd.notna(plt):
        n += 1
        if plt >= 150:  score += 0
        elif plt >= 100: score += 1
        elif plt >= 50:  score += 2
        elif plt >= 20:  score += 3
        else:            score += 4

    # ── 3. HEPÁTICO (bilirrubina mg/dL) ───────────────────────────────────
    bili = row.get('max_bilirubin')
    if pd.notna(bili):
        n += 1
        if bili < 1.2:   score += 0
        elif bili < 2.0:  score += 1
        elif bili < 6.0:  score += 2
        elif bili < 12.0: score += 3
        else:             score += 4

    # ── 4. CARDIOVASCULAR (PAM mmHg) ──────────────────────────────────────
    # Sin información de vasopresores: PAM >= 70 → 0 puntos; PAM < 70 → 1 punto (mínimo)
    map_val = compute_map(row)
    if pd.notna(map_val):
        n += 1
        score += 0 if map_val >= 70 else 1

    # ── 5. NEUROLÓGICO (GCS): NO DISPONIBLE ───────────────────────────────

    # ── 6. RENAL (creatinina mg/dL) ───────────────────────────────────────
    creat = row.get('max_creatinine')
    if pd.notna(creat):
        n += 1
        if creat < 1.2:   score += 0
        elif creat < 2.0:  score += 1
        elif creat < 3.5:  score += 2
        elif creat < 5.0:  score += 3
        else:              score += 4

    return pd.Series({'sofa_partial': score if n > 0 else np.nan,
                      'sofa_n_components': n})


print('Calculando SOFA parcial...')
sofa_result = df.apply(compute_sofa_partial, axis=1)
df = pd.concat([df, sofa_result], axis=1)

print(f"SOFA calculado. Cobertura: {df['sofa_partial'].notna().mean():.1%}")
print(df['sofa_partial'].describe().round(2))
print(f"\nComponentes disponibles por paciente (mediana): {df['sofa_n_components'].median()}")

Calculando SOFA parcial...
SOFA calculado. Cobertura: 94.4%
count    3690.00
mean        1.57
std         1.87
min         0.00
25%         0.00
50%         1.00
75%         3.00
max        11.00
Name: sofa_partial, dtype: float64

Componentes disponibles por paciente (mediana): 2.0


## 12. Cálculo del APACHE II parcial (11/12 componentes)

**Componentes calculados:**
Temperatura, PAM, FC, FR, Oxigenación, pH arterial, Na, K, Creatinina, Hematocrito, WBC, Edad

**No calculados por falta de datos:**
- GCS (no disponible en datos estructurados)
- Puntos por enfermedad crónica (requeriría análisis ICD completo)

> **Nota metodológica:** El APACHE II estándar tiene un máximo de 71 puntos. Esta versión parcial, sin GCS (máx. 15) ni crónicas (máx. 5), tiene un máximo teórico de 51 puntos. Se documenta como limitación.

In [21]:
def compute_apache2_partial(row):
    """
    APACHE II parcial: sin GCS ni puntos por enfermedad crónica.
    Temperatura en °F → convertida a °C internamente.
    Para Na y K se usa el PEOR valor (max para hiperNa/hiperK, min para hipoNa/hipoK).
    
    Returns: (score, n_components_disponibles)
    """
    score = 0
    n = 0

    # ── 1. TEMPERATURA (°F → °C) ───────────────────────────────────────────
    temp_f = row.get('max_temp_f')
    if pd.notna(temp_f):
        t = (temp_f - 32) * 5 / 9
        n += 1
        if t >= 41:             score += 4
        elif t >= 39:           score += 1
        elif t >= 36:           score += 0
        elif t >= 34:           score += 1
        elif t >= 32:           score += 2
        elif t >= 30:           score += 3
        else:                   score += 4

    # ── 2. PRESIÓN ARTERIAL MEDIA (mmHg) ──────────────────────────────────
    map_val = compute_map(row)
    if pd.notna(map_val):
        n += 1
        if map_val >= 160:       score += 4
        elif map_val >= 130:     score += 3
        elif map_val >= 110:     score += 2
        elif map_val >= 70:      score += 0
        elif map_val >= 50:      score += 2
        else:                    score += 4

    # ── 3. FRECUENCIA CARDÍACA (lpm) ──────────────────────────────────────
    hr = row.get('max_pulse')
    if pd.notna(hr):
        n += 1
        if hr >= 180:            score += 4
        elif hr >= 140:          score += 3
        elif hr >= 110:          score += 2
        elif hr >= 70:           score += 0
        elif hr >= 55:           score += 2
        elif hr >= 40:           score += 3
        else:                    score += 4

    # ── 4. FRECUENCIA RESPIRATORIA (rpm) ──────────────────────────────────
    rr = row.get('max_resp_rate')
    if pd.notna(rr):
        n += 1
        if rr >= 50:             score += 4
        elif rr >= 35:           score += 3
        elif rr >= 25:           score += 1
        elif rr >= 12:           score += 0
        elif rr >= 10:           score += 1
        elif rr >= 6:            score += 2
        else:                    score += 4

    # ── 5. OXIGENACIÓN (PaO₂ mmHg o SpO₂ proxy) ──────────────────────────
    # APACHE II: si FiO₂ < 0.5 → usar PaO₂; si FiO₂ >= 0.5 → usar AaDO₂ (no calculable sin PaCO₂)
    # Estrategia: para ventilados usamos SpO₂-proxy; para no ventilados usamos PaO₂
    pao2 = row.get('min_pao2')
    spo2 = row.get('min_oxygen_saturation')
    is_vent = row.get('is_mechanical_ventilation', 0) == 1

    if pd.notna(pao2) and not is_vent:
        n += 1
        if pao2 >= 70:           score += 0
        elif pao2 >= 61:         score += 1
        elif pao2 >= 55:         score += 3
        else:                    score += 4
    elif pd.notna(spo2):
        # SpO₂ proxy: relación no lineal, usar puntos de corte empíricos validados
        n += 1
        if spo2 >= 95:           score += 0   # PaO₂ probable >= 70
        elif spo2 >= 91:         score += 1   # PaO₂ probable 61-70
        elif spo2 >= 85:         score += 3   # PaO₂ probable 55-60
        else:                    score += 4   # PaO₂ probable < 55

    # ── 6. pH ARTERIAL ────────────────────────────────────────────────────
    # Usar el peor valor: mínimo (acidosis) tiene más impacto clínico
    ph = row.get('min_ph')
    if pd.notna(ph):
        n += 1
        if ph >= 7.70:           score += 4
        elif ph >= 7.60:         score += 3
        elif ph >= 7.50:         score += 1
        elif ph >= 7.33:         score += 0
        elif ph >= 7.25:         score += 2
        elif ph >= 7.15:         score += 3
        else:                    score += 4

    # ── 7. SODIO SÉRICO (mEq/L) — peor valor ─────────────────────────────
    # El peor de min_sodium vs max_sodium (hipoNa < 130 o hiperNa > 150)
    na_min = row.get('min_sodium')
    na_max = row.get('max_sodium')
    na = None
    if pd.notna(na_min) and pd.notna(na_max):
        # Elegir el que da mayor puntuación (peor caso)
        na = na_min if na_min < 130 else na_max
    elif pd.notna(na_max):
        na = na_max
    elif pd.notna(na_min):
        na = na_min

    if na is not None:
        n += 1
        if na >= 180:            score += 4
        elif na >= 160:          score += 3
        elif na >= 155:          score += 2
        elif na >= 150:          score += 1
        elif na >= 130:          score += 0
        elif na >= 120:          score += 2
        elif na >= 111:          score += 3
        else:                    score += 4

    # ── 8. POTASIO SÉRICO (mEq/L) — peor valor ───────────────────────────
    k_min = row.get('min_potassium')
    k_max = row.get('max_potassium')
    k = None
    if pd.notna(k_min) and pd.notna(k_max):
        k = k_max if k_max >= 5.5 else k_min
    elif pd.notna(k_max):
        k = k_max
    elif pd.notna(k_min):
        k = k_min

    if k is not None:
        n += 1
        if k >= 7.0:             score += 4
        elif k >= 6.0:           score += 3
        elif k >= 5.5:           score += 1
        elif k >= 3.5:           score += 0
        elif k >= 3.0:           score += 1
        elif k >= 2.5:           score += 2
        else:                    score += 4

    # ── 9. CREATININA (mg/dL) — x2 si insuficiencia renal aguda ──────────
    creat = row.get('max_creatinine')
    if pd.notna(creat):
        n += 1
        if creat >= 3.5:         pts = 4
        elif creat >= 2.0:       pts = 3
        elif creat >= 1.5:       pts = 2
        elif creat >= 0.6:       pts = 0
        else:                    pts = 2
        # CRRT como proxy de insuficiencia renal aguda (dobla puntuación, máx 4)
        if row.get('is_crrt', 0) == 1:
            pts = min(pts * 2, 4)
        score += pts

    # ── 10. HEMATOCRITO (%) — peor valor ─────────────────────────────────
    hct = row.get('max_hematocrit')  # policitemia
    hct_min = row.get('min_hematocrit')  # anemia
    # Elegir el que da mayor puntuación
    hct_worst = None
    pts_hct_max = 0
    pts_hct_min = 0
    if pd.notna(hct):
        if hct >= 60: pts_hct_max = 4
        elif hct >= 50: pts_hct_max = 2
        elif hct >= 46: pts_hct_max = 1
    if pd.notna(hct_min):
        if hct_min < 20: pts_hct_min = 4
        elif hct_min < 30: pts_hct_min = 2
    if pd.notna(hct) or pd.notna(hct_min):
        n += 1
        score += max(pts_hct_max, pts_hct_min)

    # ── 11. WBC (×10³/µL) ─────────────────────────────────────────────────
    wbc = row.get('max_wbc')
    if pd.notna(wbc):
        n += 1
        if wbc >= 40:            score += 4
        elif wbc >= 20:          score += 2
        elif wbc >= 15:          score += 1
        elif wbc >= 3:           score += 0
        elif wbc >= 1:           score += 2
        else:                    score += 4

    # ── 12. GCS: NO DISPONIBLE ────────────────────────────────────────────

    # ── 13. EDAD ──────────────────────────────────────────────────────────
    age = row.get('anchor_age')
    if pd.notna(age):
        n += 1
        if age < 45:             score += 0
        elif age < 55:           score += 2
        elif age < 65:           score += 3
        elif age < 75:           score += 5
        else:                    score += 6

    # ── 14. ENFERMEDAD CRÓNICA: NO CALCULADO (requiere análisis ICD) ──────

    return pd.Series({'apache2_partial': score if n >= 3 else np.nan,
                      'apache2_n_components': n})


print('Calculando APACHE II parcial...')
apache_result = df.apply(compute_apache2_partial, axis=1)
df = pd.concat([df, apache_result], axis=1)

print(f"APACHE II calculado. Cobertura: {df['apache2_partial'].notna().mean():.1%}")
print(df['apache2_partial'].describe().round(2))
print(f"\nComponentes disponibles por paciente (mediana): {df['apache2_n_components'].median()}")

Calculando APACHE II parcial...
APACHE II calculado. Cobertura: 91.8%
count    3588.00
mean        9.31
std         4.31
min         0.00
25%         6.00
50%         9.00
75%        12.00
max        29.00
Name: apache2_partial, dtype: float64

Componentes disponibles por paciente (mediana): 6.0


### 12.1 Distribución de scores por mortalidad (validación clínica)

In [22]:
df['died_30d'] = df['dod_within_30_days'].notna().astype(int)

for score_col in ['sofa_partial', 'apache2_partial']:
    alive = df.loc[df['died_30d'] == 0, score_col].dropna()
    dead  = df.loc[df['died_30d'] == 1, score_col].dropna()
    print(f'\n{score_col}:')
    print(f'  Vivos  — media: {alive.mean():.2f} ± {alive.std():.2f} | mediana: {alive.median():.1f}')
    print(f'  Muertos — media: {dead.mean():.2f} ± {dead.std():.2f} | mediana: {dead.median():.1f}')

df.drop(columns=['died_30d'], inplace=True)


sofa_partial:
  Vivos  — media: 1.39 ± 1.73 | mediana: 1.0
  Muertos — media: 2.14 ± 2.17 | mediana: 1.0

apache2_partial:
  Vivos  — media: 8.60 ± 3.94 | mediana: 8.0
  Muertos — media: 11.61 ± 4.63 | mediana: 11.0


## 13. Indicadoras de missingness y severity_score actualizado

Se crean indicadoras binarias para todos los labs continuos. El `severity_score` se actualiza para incluir las nuevas variables fisiológicas disponibles.

In [23]:
# Columnas labs sobre las que crear indicadoras
LAB_COLS = [
    'max_lactate', 'max_creatinine', 'max_bilirubin', 'min_platelets',
    'max_bun', 'min_pao2', 'max_glucose',
    # Nuevas v6
    'min_ph', 'min_sodium', 'max_sodium', 'min_potassium', 'max_potassium',
    'min_hematocrit', 'max_hematocrit', 'min_wbc', 'max_wbc',
    'max_peep', 'max_tidal_volume',
    # Baja cobertura pero se incluye indicadora
    'max_pt', 'max_crp', 'max_ferritin', 'max_dimer',
    'min_lymphocytes', 'max_neutrophils', 'max_troponin',
]

print('Creando indicadoras de missingness...')
for col in LAB_COLS:
    if col in df.columns:
        df[f'{col}_measured'] = df[col].notna().astype(int)

print('\nCobertura labs (% no-nulos):')
cov = df[LAB_COLS].notna().mean().sort_values(ascending=False)
for c, v in cov.items():
    bar = '█' * int(v * 20) + '░' * (20 - int(v * 20))
    print(f'  {c:<30} {bar} {v:.1%}')

Creando indicadoras de missingness...

Cobertura labs (% no-nulos):
  max_glucose                    █████████░░░░░░░░░░░ 48.1%
  min_potassium                  ██████░░░░░░░░░░░░░░ 34.8%
  max_potassium                  ██████░░░░░░░░░░░░░░ 34.8%
  max_creatinine                 ██████░░░░░░░░░░░░░░ 34.7%
  max_sodium                     ██████░░░░░░░░░░░░░░ 34.3%
  min_sodium                     ██████░░░░░░░░░░░░░░ 34.3%
  max_bun                        ██████░░░░░░░░░░░░░░ 33.9%
  min_pao2                       ██████░░░░░░░░░░░░░░ 30.8%
  max_hematocrit                 █████░░░░░░░░░░░░░░░ 25.6%
  min_hematocrit                 █████░░░░░░░░░░░░░░░ 25.6%
  min_platelets                  ████░░░░░░░░░░░░░░░░ 24.6%
  min_wbc                        ████░░░░░░░░░░░░░░░░ 24.6%
  max_wbc                        ████░░░░░░░░░░░░░░░░ 24.6%
  max_bilirubin                  ████░░░░░░░░░░░░░░░░ 22.5%
  max_crp                        ███░░░░░░░░░░░░░░░░░ 18.6%
  max_ferritin                  

In [24]:
def compute_severity_score_v6(row, low_coverage_cols):
    """
    Severity score compuesto v6.
    Añade pH, Na, K, Hct y WBC con sus pesos clínicos.
    Mantiene compatibilidad con versiones anteriores.
    """
    components, weights = [], []

    def add(val, weight, norm_fn):
        if pd.notna(val):
            components.append(np.clip(norm_fn(val), 0, 1))
            weights.append(weight)

    # Respiratorio
    add(row.get('min_oxygen_saturation'), 3, lambda x: (100 - x) / 100)
    if 'min_pao2' not in low_coverage_cols:
        add(row.get('min_pao2'), 3, lambda x: 1 - x / 500)
    add(row.get('max_resp_rate'), 2, lambda x: (x - 12) / 28)

    # Hemodinámico
    add(row.get('min_bp_systolic'), 2, lambda x: 1 - x / 120)

    # Metabólico
    if 'max_lactate' not in low_coverage_cols:
        add(row.get('max_lactate'), 3, lambda x: x / 10)

    # Acidosis (nuevo v6)
    if 'min_ph' not in low_coverage_cols:
        add(row.get('min_ph'), 3, lambda x: (7.45 - x) / 0.30)  # desviación desde normal

    # Renal
    if 'max_creatinine' not in low_coverage_cols:
        add(row.get('max_creatinine'), 2, lambda x: x / 10)

    # Electrolitos (nuevo v6)
    if 'min_sodium' not in low_coverage_cols:
        na_min = row.get('min_sodium')
        na_max = row.get('max_sodium')
        na_worst = None
        if pd.notna(na_min) and pd.notna(na_max):
            na_worst = na_min if abs(na_min - 140) > abs(na_max - 140) else na_max
        elif pd.notna(na_min): na_worst = na_min
        elif pd.notna(na_max): na_worst = na_max
        if na_worst is not None:
            add(abs(na_worst - 140), 1, lambda x: x / 30)  # desviación desde 140

    # Hepático
    if 'max_bilirubin' not in low_coverage_cols:
        add(row.get('max_bilirubin'), 1, lambda x: x / 20)

    # Coagulación
    if 'min_platelets' not in low_coverage_cols:
        add(row.get('min_platelets'), 2, lambda x: 1 - x / 400)
    if 'max_dimer' not in low_coverage_cols:
        add(row.get('max_dimer'), 2, lambda x: x / 10)

    # Inflamación
    if 'max_crp' not in low_coverage_cols:
        add(row.get('max_crp'), 2, lambda x: x / 300)
    if 'min_lymphocytes' not in low_coverage_cols:
        add(row.get('min_lymphocytes'), 2, lambda x: 1 - x / 2000)
    if 'max_ferritin' not in low_coverage_cols:
        add(row.get('max_ferritin'), 2, lambda x: x / 5000)

    # Daño cardíaco
    if 'max_troponin' not in low_coverage_cols:
        add(row.get('max_troponin'), 2, lambda x: x / 10)

    # Procedimientos invasivos
    for proc, w in [('is_ecmo', 5), ('is_mechanical_ventilation', 4),
                    ('is_prone_position', 3), ('is_crrt', 3),
                    ('has_hemodialysis', 3), ('is_intubated', 2)]:
        val = row.get(proc)
        if pd.notna(val):
            components.append(float(val))
            weights.append(w)

    return np.average(components, weights=weights) if components else np.nan


# Calcular cobertura para determinar low_coverage_cols
missing_rates = df[LAB_COLS].isna().mean()
low_coverage_cols = set(missing_rates[missing_rates >= 0.70].index.tolist())
print(f'Columnas con >70% missings (excluidas del severity_score): {low_coverage_cols}')

print('\nCalculando severity_score v6...')
df['severity_score'] = df.apply(
    lambda r: compute_severity_score_v6(r, low_coverage_cols), axis=1
)
print(f"severity_score cobertura: {df['severity_score'].notna().mean():.1%}")

Columnas con >70% missings (excluidas del severity_score): {'min_lymphocytes', 'max_bilirubin', 'max_hematocrit', 'min_platelets', 'max_peep', 'min_hematocrit', 'max_crp', 'max_wbc', 'max_dimer', 'max_neutrophils', 'max_troponin', 'min_ph', 'max_ferritin', 'max_tidal_volume', 'max_lactate', 'min_wbc', 'max_pt'}

Calculando severity_score v6...
severity_score cobertura: 100.0%


## 14. Limpieza final de columnas

In [25]:
# Eliminar columnas de PA invasiva (ruidosas) y length_of_stay (leakage)
DROP_ALWAYS = ['min_bp_systolic_line', 'min_bp_diastolic_line', 'length_of_stay', 'is_dnr']
df.drop(columns=[c for c in DROP_ALWAYS if c in df.columns], inplace=True)

# Eliminar columnas intermedias de cálculo de scores
df.drop(columns=['sofa_n_components', 'apache2_n_components'], inplace=True, errors='ignore')

print(f'Shape final all_stays: {df.shape}')
print(f'Columnas ({len(df.columns)}):')
for i, c in enumerate(df.columns, 1):
    print(f'  {i:>2}. {c}')

Shape final all_stays: (3908, 80)
Columnas (80):
   1. subject_id
   2. admission_id
   3. stay_id
   4. admittime
   5. min_bp_systolic
   6. min_bp_diastolic
   7. min_oxygen_saturation
   8. max_pulse
   9. max_resp_rate
  10. max_temp_f
  11. bmi
  12. min_pao2
  13. min_platelets
  14. max_bilirubin
  15. max_creatinine
  16. min_ph
  17. max_ph
  18. min_sodium
  19. max_sodium
  20. min_potassium
  21. max_potassium
  22. min_hematocrit
  23. max_hematocrit
  24. min_wbc
  25. max_wbc
  26. max_peep
  27. max_tidal_volume
  28. max_lactate
  29. max_bun
  30. max_glucose
  31. max_pt
  32. max_crp
  33. max_ferritin
  34. max_dimer
  35. min_lymphocytes
  36. max_neutrophils
  37. max_troponin
  38. is_mechanical_ventilation
  39. is_crrt
  40. has_arterial_line
  41. is_intubated
  42. is_prone_position
  43. has_chest_tube
  44. has_central_line
  45. has_hemodialysis
  46. has_niv
  47. is_ecmo
  48. gender
  49. anchor_age
  50. dod_within_30_days
  51. marital_status
  52. 

## 15. Compactación: una estancia por paciente

**Opción B:** Estancia con mayor `severity_score` (caso más grave del paciente)  
**Opción C:** Última estancia por `admittime` (más reciente)

In [26]:
print(f'Pacientes únicos: {df["subject_id"].nunique()}')
print(f'Estancias totales: {len(df)}')
print(f'Estancias multi-ingreso: {len(df) - df["subject_id"].nunique()}')

# Opción B: estancia con mayor severity_score
df_optB = (
    df.sort_values(by=['subject_id', 'severity_score'], ascending=[True, False])
      .drop_duplicates(subset=['subject_id'], keep='first')
)
print(f'\nOptionB: {len(df_optB)} pacientes')

# Opción C: última estancia
df_optC = (
    df.sort_values(by=['subject_id', 'admittime'], ascending=[True, False])
      .drop_duplicates(subset=['subject_id'], keep='first')
)
print(f'OptionC: {len(df_optC)} pacientes')

Pacientes únicos: 2067
Estancias totales: 3908
Estancias multi-ingreso: 1841

OptionB: 2067 pacientes
OptionC: 2067 pacientes


## 16. Validación del dataset v6

In [27]:
print('=' * 55)
print('VALIDACIÓN DATASET V6')
print('=' * 55)

for name, d in [('OptionB', df_optB), ('OptionC', df_optC)]:
    print(f'\n── {name} ──────────────────────────────────────────')
    print(f'  Shape: {d.shape}')

    mortality = d['dod_within_30_days'].notna().mean()
    print(f'  Mortalidad 30d: {mortality:.2%}')

    for col in ['sofa_partial', 'apache2_partial', 'severity_score']:
        cov = d[col].notna().mean()
        med = d[col].median()
        print(f'  {col:<22}: {cov:.1%} cobertura | mediana = {med:.1f}')

    # Verificar que los scores discriminan (mayor score → mayor mortalidad)
    for score in ['sofa_partial', 'apache2_partial']:
        alive = d.loc[d['dod_within_30_days'].isna(), score].dropna()
        dead  = d.loc[d['dod_within_30_days'].notna(), score].dropna()
        direction = '✅' if dead.median() > alive.median() else '⚠️'
        print(f'  {direction} {score}: vivos={alive.median():.1f} vs muertos={dead.median():.1f}')

    # Verificar nuevas variables presentes
    for col in ['min_ph', 'min_sodium', 'min_potassium', 'max_hematocrit', 'max_wbc']:
        assert col in d.columns, f'ERROR: {col} no encontrado en {name}'
    print(f'  ✅ Nuevas variables v6 presentes')

print('\n✅ Validación completada.')

VALIDACIÓN DATASET V6

── OptionB ──────────────────────────────────────────
  Shape: (2067, 80)
  Mortalidad 30d: 23.37%
  sofa_partial          : 95.8% cobertura | mediana = 1.0
  apache2_partial       : 93.8% cobertura | mediana = 9.0
  severity_score        : 100.0% cobertura | mediana = 0.1
  ✅ sofa_partial: vivos=1.0 vs muertos=2.0
  ✅ apache2_partial: vivos=8.0 vs muertos=12.0
  ✅ Nuevas variables v6 presentes

── OptionC ──────────────────────────────────────────
  Shape: (2067, 80)
  Mortalidad 30d: 23.51%
  sofa_partial          : 95.2% cobertura | mediana = 1.0
  apache2_partial       : 93.3% cobertura | mediana = 9.0
  severity_score        : 100.0% cobertura | mediana = 0.1
  ✅ sofa_partial: vivos=1.0 vs muertos=2.0
  ✅ apache2_partial: vivos=8.0 vs muertos=12.0
  ✅ Nuevas variables v6 presentes

✅ Validación completada.


## 17. Guardado de datasets v6

In [ ]:
df_optB.to_parquet('../datasets/v6/dataset_v6_optionB.parquet', index=False, compression='snappy')
df_optB.to_csv('../datasets/v6/dataset_v6_optionB.csv', index=False)
print(f'✅ OptionB guardado: {df_optB.shape}')

df_optC.to_parquet('../datasets/v6/dataset_v6_optionC.parquet', index=False, compression='snappy')
df_optC.to_csv('../datasets/v6/dataset_v6_optionC.csv', index=False)
print(f'✅ OptionC guardado: {df_optC.shape}')

print('\nResumen final:')
print(f'  Parquet all_stays → data/processed/covid_icu_dataset_v6_all_stays.parquet')
print(f'  Dataset final     → datasets/v6/dataset_v6_optionB.parquet')
print(f'                    → datasets/v6/dataset_v6_optionC.parquet')

✅ OptionB guardado: (2067, 80)
✅ OptionC guardado: (2067, 80)

Resumen final:
  Parquet all_stays → data/processed/covid_icu_dataset_v6_all_stays.parquet
  Dataset final     → datasets/v6/dataset_v6_optionB.parquet
                    → datasets/v6/dataset_v6_optionC.parquet
